# Análise de dados TCP-CII

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('./T CELL/DENV 2 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,42,0.08,WKIEKASFI,0.970218,0.08
1,1,GSGIFITDNVHTWTE,16,30,15,HLA-DQA1*05:01/DQB1*02:01,4,0.24,FITDNVHTW,0.708095,0.24
2,1,QYKFQPESPSKLASA,31,45,15,HLA-DRB1*01:01,7,0.32,FQPESPSKL,0.923397,0.32
3,1,AAIKDNRAVHADMGY,186,200,15,HLA-DRB3*02:02,38,0.32,IKDNRAVHA,0.712484,0.32
4,1,TNRAWNSLEVEDYGF,146,160,15,HLA-DQA1*05:01/DQB1*02:01,30,0.37,WNSLEVEDY,0.678171,0.37
...,...,...,...,...,...,...,...,...,...,...,...
1831,1,KLITEWCCRSCTLPP,306,320,15,HLA-DRB1*04:01,62,100.00,EWCCRSCTL,0.000066,100.00
1832,1,VVTEDCGNRGPSLRT,286,300,15,HLA-DQA1*03:01/DQB1*03:02,58,100.00,VTEDCGNRG,0.000061,100.00
1833,1,CTLPPLRYRGEDGCW,316,330,15,HLA-DQA1*03:01/DQB1*03:02,64,100.00,LRYRGEDGC,0.000048,100.00
1834,1,TTASGKLITEWCCRS,301,315,15,HLA-DRB3*02:02,61,100.00,LITEWCCRS,0.000033,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [11]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,42,0.08,WKIEKASFI,0.970218,0.08
1,1,GSGIFITDNVHTWTE,16,30,15,HLA-DQA1*05:01/DQB1*02:01,4,0.24,FITDNVHTW,0.708095,0.24
2,1,QYKFQPESPSKLASA,31,45,15,HLA-DRB1*01:01,7,0.32,FQPESPSKL,0.923397,0.32
3,1,AAIKDNRAVHADMGY,186,200,15,HLA-DRB3*02:02,38,0.32,IKDNRAVHA,0.712484,0.32
4,1,TNRAWNSLEVEDYGF,146,160,15,HLA-DQA1*05:01/DQB1*02:01,30,0.37,WNSLEVEDY,0.678171,0.37
...,...,...,...,...,...,...,...,...,...,...,...
59,1,GVFTTNIWLKLKEKQ,161,175,15,HLA-DPA1*02:01/DPB1*05:01,33,4.60,FTTNIWLKL,0.075697,4.60
60,1,ADMGYWIESALNDTW,196,210,15,HLA-DPA1*01:03/DPB1*04:01,40,4.60,YWIESALND,0.061374,4.60
61,1,AAIKDNRAVHADMGY,186,200,15,HLA-DRB1*13:02,38,4.80,IKDNRAVHA,0.325697,4.80
62,1,NRAVHADMGYWIESA,191,205,15,HLA-DRB1*03:01,39,4.80,VHADMGYWI,0.308584,4.80


## Agrupando por pepitideos e agregando colunas pertinentes.

In [12]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAVHADMGY,186,200,3,3.30,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*13:02, HLA..."
1,ADMGYWIESALNDTW,196,210,2,3.05,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*04:05"
2,DFDFCDGTTVVVTED,276,290,1,4.80,HLA-DQA1*03:01/DQB1*03:02
3,DGPETAECPNTNRAW,136,150,1,3.70,HLA-DQA1*05:01/DQB1*03:01
4,DSGCVVSWKNKELKC,1,15,1,0.56,HLA-DRB1*15:01
5,ENEVKLTIMTGDIKG,81,95,1,4.30,HLA-DRB5*01:01
6,GSGIFITDNVHTWTE,16,30,7,3.10,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
7,GVFTTNIWLKLKEKQ,161,175,5,1.00,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
8,ITDNVHTWTEQYKFQ,21,35,2,4.25,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*04:01/DQB1..."
9,ITPELNHILSENEVK,71,85,1,2.80,HLA-DRB1*04:05


# Filtragem por epítopos que presentes em mais de 2 alelos.

In [13]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 2
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAVHADMGY,186,200,3,3.30,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*13:02, HLA..."
1,ADMGYWIESALNDTW,196,210,2,3.05,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*04:05"
2,GSGIFITDNVHTWTE,16,30,7,3.10,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
3,GVFTTNIWLKLKEKQ,161,175,5,1.00,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
4,ITDNVHTWTEQYKFQ,21,35,2,4.25,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*04:01/DQB1..."
5,KRSLRPQPTELKYSW,101,115,2,1.35,"HLA-DRB4*01:01, HLA-DRB5*01:01"
6,LKYSWKTWGKAKMLS,111,125,2,2.75,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
7,LNDTWKIEKASFIEV,206,220,9,2.20,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
8,NRAVHADMGYWIESA,191,205,2,2.95,"HLA-DRB1*03:01, HLA-DRB3*01:01"
9,QTFLIDGPETAECPN,131,145,3,3.00,"HLA-DRB1*04:01, HLA-DRB3*01:01, HLA-DRB3*02:02"


In [14]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,GVFTTNIWLKLKEKQ,161,175,5,1.00,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,KRSLRPQPTELKYSW,101,115,2,1.35,"HLA-DRB4*01:01, HLA-DRB5*01:01"
2,TNRAWNSLEVEDYGF,146,160,4,1.50,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
3,YRPGYHTQITGPWHL,256,270,2,1.70,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
4,QYKFQPESPSKLASA,31,45,7,2.00,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1..."
5,LNDTWKIEKASFIEV,206,220,9,2.20,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
6,LKYSWKTWGKAKMLS,111,125,2,2.75,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
7,NRAVHADMGYWIESA,191,205,2,2.95,"HLA-DRB1*03:01, HLA-DRB3*01:01"
8,QTFLIDGPETAECPN,131,145,3,3.00,"HLA-DRB1*04:01, HLA-DRB3*01:01, HLA-DRB3*02:02"
9,ADMGYWIESALNDTW,196,210,2,3.05,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*04:05"
